In [ ]:
import os

# Vast.ai HF credential location
os.environ["HF_HOME"] = "/workspace/.hf_home"

from huggingface_hub import login, get_token

hf_token = get_token()

if hf_token:
    print("Using cached Hugging Face authentication.")
else:
    raise RuntimeError("Hugging Face token not found.")

Hugging Face token not found. Add it in Colab Secrets.


In [ ]:
import sys

!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install numpy scikit-learn


!{sys.executable} -m pip install git+https://github.com/AIPHES/emnlp19-moverscore.git

!{sys.executable} -m pip install accelerate rouge-score bert-score nltk torch torchvision pyemd datasets transformers peft "torchao>=0.16.0"

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 13.1 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 45.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 70.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]
  Cloning https://github.com/AIPHES/emnlp19-moverscore.git to ./tmp/pip-req-build-3c9lx58o
  Running command git clone --filter=blob:none --quiet https://github.com/AIPHES/emnlp19-moverscore.git /tmp/pip-req-build-3c9lx58o
  Re

True

In [ ]:
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

import torch
import gc
import torch.nn as nn
import math
import os
import json
import threading
from typing import Any, Dict, Optional, Tuple, List
from torch.optim import Optimizer
from PIL import Image
from io import BytesIO
import requests
SEED = 42

# ---------------------------------------------------------------------------
# RUN MODE
#   SMOKE_TEST = True  -> 32 / 4 / 4 examples, smoke GRIT hyperparameters
#                         (24 optimizer steps; K-FAC inverts at steps 10, 20)
#   SMOKE_TEST = False -> 8000 / 1000 / 1000 full protocol
# ---------------------------------------------------------------------------
SMOKE_TEST = False

RUN_NAME = f"GRIT_llama8_seed{SEED}_VG" + ("_smoke" if SMOKE_TEST else "")
OUTPUT_DIR = os.path.abspath(RUN_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"All experiment outputs will be saved in:\n{OUTPUT_DIR}")

if "hf_token" not in globals():
    hf_token = os.getenv("HF_TOKEN")

import re
import numpy as np
# Patch for numpy.float deprecation in MoverScore
if not hasattr(np, 'float'):
    np.float = float

from transformers import EvalPrediction
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.metrics import f1_score
import nltk
nltk.download('punkt', quiet=True)


from datasets import load_dataset
from transformers import (
    MllamaForConditionalGeneration,
    AutoProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
    set_seed,
)
from moverscore_v2 import get_idf_dict, word_mover_score
from peft import LoraConfig, get_peft_model


# Instruction used for BOTH train and eval: lmms-eval's base RefCOCO
# instruction + our brevity suffix (identical to the Gemma run).
REFCOCO_INSTRUCTION = (
    "Provide a short description for this region. "
    "Respond with only the short description itself "
    "(a few words), with no explanation or extra text."
)

# Keys the collator puts in the batch that are metadata, NOT model inputs.
# Gemma's forward silently swallowed unknown kwargs; Mllama forwards **kwargs
# down into the text model, so these must be stripped before every forward.
NON_MODEL_KEYS = ("refcoco_answers", "prompt_length")


def load_image_from_source(image_source):
    """
    Robust image loading for RefCOCO examples.

    lmms-lab-encoder/RefCOCO's `image` column is already a decoded
    PIL.Image (the raw COCO image, with NO region visualization drawn on
    it — see draw_region_indicator() below for that). We keep the
    URL/path/bytes fallbacks from the VQAv2 version defensively, in case
    a different source ever gets swapped in, but they aren't expected to
    be exercised for this dataset.
    """
    try:
        if isinstance(image_source, Image.Image):
            return image_source.convert('RGB') if image_source.mode != 'RGB' else image_source

        if isinstance(image_source, str):
            if image_source.startswith(('http://', 'https://')):
                response = requests.get(image_source, timeout=10)
                response.raise_for_status()
                image = Image.open(BytesIO(response.content))
                return image.convert('RGB') if image.mode != 'RGB' else image

            if os.path.exists(image_source):
                image = Image.open(image_source)
                return image.convert('RGB') if image.mode != 'RGB' else image

        if isinstance(image_source, bytes):
            image = Image.open(BytesIO(image_source))
            return image.convert('RGB') if image.mode != 'RGB' else image

        return None

    except Exception as e:
        print(f"Failed to load image: {e}")
        return None


def draw_region_indicator(image, bbox, color=(255, 0, 0), width_frac=0.006):
    """
    Draw a red RECTANGLE around the referred region.

    lmms-eval's own reference implementation for this exact dataset
    (lmms_eval/tasks/refcoco/utils.py, refcoco_bbox_doc_to_visual) draws a
    plain rectangle at the *exact* bbox, no padding, no circle:

        bbox_xy = [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
        draw.rectangle(bbox_xy, outline="red")

    We mirror that (bbox is COCO-style [x, y, w, h] in pixel coordinates,
    confirmed against the dataset's own schema) so the visual indicator
    matches what the harness/paper convention actually is, and so the
    prompt's wording ("the region" / "the box", not "circled") matches
    what's drawn. We keep a slightly thicker line than the harness's
    default 1px, since a 1px line is prone to disappearing once the image
    gets downsampled by the vision processor before the model ever sees it
    — the harness only ever runs zero-shot inference on the original-size
    image, so it doesn't have to worry about that, but we do.
    """
    from PIL import ImageDraw

    annotated = image.copy()
    if not bbox or len(bbox) != 4:
        return annotated

    x, y, w, h = bbox
    if w <= 0 or h <= 0:
        return annotated

    # Origin format (top x, top y, width, height) -> (x0, y0, x1, y1),
    # exactly as lmms-eval's refcoco_bbox_doc_to_visual does it. No padding.
    x0, y0 = x, y
    x1 = min(annotated.width, x + w)
    y1 = min(annotated.height, y + h)

    line_width = max(2, int(round(min(annotated.width, annotated.height) * width_frac)))

    draw = ImageDraw.Draw(annotated)
    draw.rectangle([x0, y0, x1, y1], outline=color, width=line_width)
    return annotated


def extract_raw_answers(example):
    """
    Return the list of raw human reference-expression strings for a
    RefCOCO example, for use with the soft-accuracy metric.

    Unlike VQAv2's dict-based `answers` field, RefCOCO's `answer` field is
    already a flat list of plain strings (the 2-4 human referring
    expressions collected for that region). We still tolerate a
    dict-based format defensively in case a different mirror is swapped
    in later.
    """
    answers_field = example.get('answer')
    if not answers_field:
        return []
    raw = []
    for a in answers_field:
        ans = a.get('answer') if isinstance(a, dict) else a
        if ans:
            raw.append(str(ans))
    return raw


class KFACAutogradFunction(torch.autograd.Function):
    """Custom autograd function to capture activations and gradients for K-FAC."""

    @staticmethod
    def forward(ctx, module, output, input):
        ctx.module = module
        ctx.save_for_backward(input.detach())
        return output

    @staticmethod
    def backward(ctx, grad_wrt_output):
        module = ctx.module
        input, = ctx.saved_tensors
        manager = module.grit_manager

        if not module.training:
            return None, grad_wrt_output, None

        tick = manager.cov_update_tick.get(module, 0) + 1
        manager.cov_update_tick[module] = tick
        if tick % manager.config.grit_cov_update_freq != 0:
            return None, grad_wrt_output, None

        with torch.no_grad():
            a = input.reshape(-1, input.shape[-1])
            if a.shape[0] > 0 and 'default' in module.lora_A:
                lora_A_T = module.lora_A['default'].weight.data.T.to(device=a.device, dtype=a.dtype, non_blocking=True)
                projected_a = a @ lora_A_T
                a_cov_sample = projected_a.T @ projected_a

                current_cov = manager.a_covs[module].to(a_cov_sample.device)
                n = manager.num_samples_a[module]
                new_n = n + projected_a.shape[0]
                if new_n > 0:
                    updated_cov = (current_cov.float() * n + a_cov_sample.float()) / new_n
                    manager.a_covs[module].copy_(updated_cov.to(dtype=manager.a_covs[module].dtype))
                    manager.num_samples_a[module] = new_n

            g = grad_wrt_output.reshape(-1, grad_wrt_output.shape[-1])
            if g.shape[0] > 0 and 'default' in module.lora_B:
                lora_B = module.lora_B['default'].weight.data.to(device=g.device, dtype=g.dtype, non_blocking=True)
                projected_g = g @ lora_B
                g_cov_sample = projected_g.T @ projected_g

                current_cov = manager.g_covs[module].to(g_cov_sample.device)
                n = manager.num_samples_g[module]
                new_n = n + projected_g.shape[0]
                if new_n > 0:
                    updated_cov = (current_cov.float() * n + g_cov_sample.float()) / new_n
                    manager.g_covs[module].copy_(updated_cov.to(dtype=manager.g_covs[module].dtype))
                    manager.num_samples_g[module] = new_n

        return None, grad_wrt_output, None




class GRITConfig:
    """Configuration for GRIT training on Llama-3.2-11B-Vision-Instruct (RefCOCO)."""

    def __init__(self):
        self.model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
        self.batch_size = 1
        self.gradient_accumulation_steps = 4
        self.num_epochs = 3
        self.precision = "bf16"
        # Mllama puts a single <|image|> token in the text stream (vision
        # enters through cross-attention), so prompt+answer is ~40-70 tokens.
        self.max_length = 1024

        # LoRA configuration (same scope as our other Llama experiments)
        self.lora_rank = 8
        self.lora_alpha = 16
        self.lora_dropout = 0.05

        # Language tower: self-attn AND cross-attn layers of the Llama text model
        self.lora_target_modules = [
            "q_proj",
            "k_proj",
            "v_proj",
        ]

        # Vision tower (Mllama vision encoder: local + global transformer)
        self.apply_lora_to_vision = True
        self.vision_lora_modules = [
            "q_proj",
            "k_proj",
            "v_proj",
        ]

        # GRIT hyperparameters shared by both modes
        self.kfac_damping = 0.005
        self.reprojection_k = 2
        self.use_two_sided_reprojection = True
        self.enable_rank_adaptation = True
        self.rank_adaptation_threshold = 0.99
        self.min_lora_rank = 4
        self.lambda_kfac = 5e-6
        self.lambda_reproj = 5e-5

        if SMOKE_TEST:
            self.train_size, self.val_size, self.test_size = 32, 4, 4
            self.learning_rate = 5e-5
            self.kfac_update_freq = 10
            self.kfac_min_samples = 8
            self.grit_cov_update_freq = 1
            self.reprojection_freq = 300
            self.reprojection_warmup_steps = 500
            self.rank_adaptation_start_step = 500
            self.regularizer_warmup_steps = 100
            self.ng_warmup_steps = 100
            self.logging_steps = 1
            self.warmup_steps = 1
        else:
            # Full protocol = the values the Gemma RefCOCO run actually used
            # (GRITConfig defaults + the overrides at the top of its main()).
            self.train_size, self.val_size, self.test_size = 8000, 1000, 1000
            self.learning_rate = 5e-5
            self.kfac_update_freq = 150
            self.kfac_min_samples = 256
            self.grit_cov_update_freq = 2
            self.reprojection_freq = 300
            self.reprojection_warmup_steps = 200
            self.rank_adaptation_start_step = 500
            self.regularizer_warmup_steps = 300
            self.ng_warmup_steps = 150
            self.logging_steps = 50
            self.warmup_steps = 200

        # Dataset configuration
        # lmms-lab-encoder/RefCOCO is an eval-only repackaging (val/test/testA/testB,
        # no "train"). "val" -> train, "testA" -> validation, "testB" -> test.
        self.dataset_name = "lmms-lab-encoder/RefCOCO"
        self.num_workers = 0
        self.pin_memory = True
        self.drop_last = True

        self.log_fisher_spectrum = True
        self.log_top_eigs = 8
        self.log_eig_heatmaps = True
        self.log_eigs_bar = False
        self.log_eig_heatmaps_modules = 4
        self.log_eff_rank_on_inversion = False
        self.log_final_eff_rank = True
        self.kfac_inversion_device = 'cpu'

        self.answer_max_length = 512
        self.ignore_index = -100

        # Reference expressions are a few words; cap eval generation.
        self.generation_max_new_tokens = 64


class GRITManager:
    """Manager for GRIT on Llama-3.2-11B-Vision-Instruct."""

    def __init__(self, model, config, device):
        self.model = model
        self.config = config
        self.device = device
        self.global_step = 0
        self.backward_step = 0
        self.loss_history = []
        self.loss_history_capacity = 20
        self.last_kfac_update_step = 0
        self.last_reprojection_step = 0
        self.a_covs = {}
        self.g_covs = {}
        self.a_invs = {}
        self.g_invs = {}
        self.num_samples_a = {}
        self.num_samples_g = {}
        self.cov_update_tick = {}
        self.optimized_modules = []
        self.factors_are_ready = False
        self._last_Va_k = {}
        self._last_Vg_k = {}
        self._instrument_model()
        print("GRITManager: Initialization complete for Llama-3.2-11B-Vision-Instruct.")
        print(f"Optimizing {len(self.optimized_modules)} LoRA modules across vision + language towers.")

    def _compute_effective_rank(self, cov: torch.Tensor) -> float:
        """Entropy-based effective rank: exp(H(p)), p = λ / trace(cov)."""
        try:
            with torch.no_grad():
                evals = torch.linalg.eigvalsh(cov.float())
                evals = torch.clamp(evals, min=0.0)
                total = float(evals.sum().item())
                if not math.isfinite(total) or total <= 0.0:
                    return 0.0
                p = (evals / total).clamp_min(1e-12)
                entropy = float((-(p * torch.log(p)).sum()).item())
                r_eff = math.exp(entropy)
                n = int(cov.shape[0])
                if not math.isfinite(r_eff):
                    return 0.0
                return float(max(0.0, min(n, r_eff)))
        except Exception:
            return 0.0

    def _instrument_model(self):
        """Instrument both vision and language LoRA modules (Mllama: vision_model / language_model)."""
        from peft.tuners.lora import LoraLayer
        from peft.tuners.lora.layer import Linear as LoraLinear

        vision_modules = 0
        language_modules = 0

        for name, module in self.model.named_modules():
            if isinstance(module, LoraLayer) and module.r.get('default', 0) > 0 and isinstance(module.base_layer, nn.Linear):
                module.lora_name = name
                module.grit_manager = self

                name_lower = name.lower()
                # Vision tower detection (Mllama vision encoder lives under "vision_model")
                is_vision = any([
                    'vision_tower' in name_lower,
                    'vision_model' in name_lower,
                    'vision_encoder' in name_lower,
                    'visual' in name_lower and 'language' not in name_lower,
                    name.startswith('model.vision'),
                    name.startswith('vision_tower'),
                ])

                self.num_samples_a[module] = 0
                self.num_samples_g[module] = 0
                self.cov_update_tick[module] = 0

                module.original_forward = module.forward
                def new_forward(self, x=None, *args, **kwargs):
                    if x is None:
                        x = kwargs.get("hidden_states", None)
                        if x is None and len(args) > 0:
                            x = args[0]
                    y = self.original_forward(x, *args, **kwargs)
                    return KFACAutogradFunction.apply(self, y, x)

                module.forward = new_forward.__get__(module, LoraLayer)

                r = module.r['default']
                cov_device = self.device if torch.cuda.is_available() else torch.device('cpu')
                self.a_covs[module] = torch.zeros((r, r), device=cov_device, dtype=torch.float16)
                self.g_covs[module] = torch.zeros((r, r), device=cov_device, dtype=torch.float16)
                self.optimized_modules.append(module)

                if is_vision:
                    vision_modules += 1
                else:
                    language_modules += 1

        print(f"Instrumented {vision_modules} vision modules + {language_modules} language modules")
        print(f"Total LoRA modules with K-FAC: {len(self.optimized_modules)}")

    def _get_adaptive_freq(self, base_freq, min_freq=1, max_freq=1000, window=20):
        if len(self.loss_history) < window:
            return base_freq
        recent_losses = self.loss_history[-window:]
        first_half = sum(recent_losses[:window//2]) / (window//2)
        second_half = sum(recent_losses[window//2:]) / (window//2)
        if second_half < first_half * 0.99:
            new_freq = int(base_freq * 1.5)
        else:
            new_freq = int(base_freq * 0.75)
        return max(min_freq, min(new_freq, max_freq))

    def step(self, loss=None):
        self.global_step += 1
        if loss is not None:
            self.loss_history.append(loss)
            if len(self.loss_history) > self.loss_history_capacity:
                self.loss_history = self.loss_history[-self.loss_history_capacity:]

        if len(self.loss_history) > 10:
            loss_variance = torch.tensor(self.loss_history).var().item()
            self.config.kfac_damping = max(1e-6, min(0.01, 0.001 + math.sqrt(loss_variance)))

        kfac_freq = self._get_adaptive_freq(self.config.kfac_update_freq)
        if self.global_step - self.last_kfac_update_step >= kfac_freq:
            self.update_and_invert_factors()
            self.last_kfac_update_step = self.global_step

        reproj_freq = self._get_adaptive_freq(self.config.reprojection_freq, min_freq=1, max_freq=2000)
        if self.global_step >= int(getattr(self.config, 'reprojection_warmup_steps', 0) or 0) and \
           self.global_step - self.last_reprojection_step >= reproj_freq:
            self.neural_reprojection()
            self.last_reprojection_step = self.global_step

    def update_and_invert_factors(self):
        print(f"\nGRITManager: Inverting K-FAC factors at step {self.global_step}...")
        if not hasattr(self, "_scripted_invert_fn") or self._scripted_invert_fn is None:
            self._scripted_invert_fn = torch.jit.script(jit_invert_tensor_pair)
        scripted_invert_fn = self._scripted_invert_fn

        for module in self.optimized_modules:
            min_samples = getattr(self.config, 'kfac_min_samples', 128)
            if self.num_samples_a.get(module, 0) < min_samples or self.num_samples_g.get(module, 0) < min_samples:
                continue

            inv_device = str(getattr(self.config, 'kfac_inversion_device', 'cpu') or 'cpu')
            if inv_device == 'cuda' and torch.cuda.is_available():
                device_for_inv = torch.device('cuda')
            else:
                device_for_inv = torch.device('cpu')

            a_cov = self.a_covs[module].detach().to(device=device_for_inv, dtype=torch.float32)
            g_cov = self.g_covs[module].detach().to(device=device_for_inv, dtype=torch.float32)

            a_inv, g_inv = scripted_invert_fn(a_cov=a_cov, g_cov=g_cov, kfac_damping=self.config.kfac_damping)
            if a_inv.numel() > 0 and g_inv.numel() > 0:
                self.a_invs[module] = a_inv.to('cpu')
                self.g_invs[module] = g_inv.to('cpu')

        self.factors_are_ready = True

    def log_final_effective_ranks(self):
        if not getattr(self.config, 'log_final_eff_rank', True):
            return
        try:
            for module in self.optimized_modules:
                a_cov = self.a_covs[module].detach().to(device='cpu', dtype=torch.float32)
                g_cov = self.g_covs[module].detach().to(device='cpu', dtype=torch.float32)
                r_eff_a = self._compute_effective_rank(a_cov)
                r_eff_g = self._compute_effective_rank(g_cov)
                print(f"Fisher/{module.lora_name}/eff_rank_a: {r_eff_a:.4f}")
                print(f"Fisher/{module.lora_name}/eff_rank_g: {r_eff_g:.4f}")
        except Exception:
            pass

    def log_final_raw_ranks(self):
        try:
            for module in self.optimized_modules:
                base_r = int(module.r['default']) if hasattr(module, 'r') and 'default' in module.r else 0
                Va = getattr(self, '_last_Va_k', {}).get(module, None)
                Vg = getattr(self, '_last_Vg_k', {}).get(module, None)
                k_a = int(Va.shape[1]) if Va is not None and Va.numel() > 0 else base_r
                k_b = int(Vg.shape[1]) if Vg is not None and Vg.numel() > 0 else k_a
                print(f"Fisher/{module.lora_name}/final_k_a: {k_a}")
                print(f"Fisher/{module.lora_name}/final_k_b: {k_b}")
        except Exception:
            pass

    def precondition_gradients(self):
        if not self.factors_are_ready:
            return

        with torch.no_grad():
            ng_warmup = int(getattr(self.config, 'ng_warmup_steps', 0) or 0)
            if self.global_step < ng_warmup:
                return

            _total_nat_norm_sq = torch.tensor(0.0, device=self.device)

            for module in self.optimized_modules:
                if module not in self.a_invs or module not in self.g_invs:
                    continue

                lora_a = module.lora_A['default']
                lora_b = module.lora_B['default']

                if (lora_a is None or lora_b is None or
                    lora_a.weight.grad is None or lora_b.weight.grad is None):
                    continue

                a_inv_f32 = self.a_invs[module].to(self.device, dtype=torch.float32)
                g_inv_f32 = self.g_invs[module].to(self.device, dtype=torch.float32)

                grad_b_f32 = lora_b.weight.grad.detach().to(torch.float32)
                preconditioned_b_grad_f32 = grad_b_f32 @ g_inv_f32

                grad_a_f32 = lora_a.weight.grad.detach().to(torch.float32)
                r = module.r['default']
                if r > 0:
                    grad_a_f32 = grad_a_f32.view(r, -1)
                preconditioned_a_grad_f32 = a_inv_f32 @ grad_a_f32

                _total_nat_norm_sq = _total_nat_norm_sq + preconditioned_a_grad_f32.pow(2).sum() + preconditioned_b_grad_f32.pow(2).sum()

                preconditioned_a_grad_f32 = torch.nan_to_num(preconditioned_a_grad_f32, nan=0.0, posinf=0.0, neginf=0.0)
                preconditioned_b_grad_f32 = torch.nan_to_num(preconditioned_b_grad_f32, nan=0.0, posinf=0.0, neginf=0.0)
                preconditioned_a_grad_f32 = torch.clamp(preconditioned_a_grad_f32, min=-1e6, max=1e6)
                preconditioned_b_grad_f32 = torch.clamp(preconditioned_b_grad_f32, min=-1e6, max=1e6)

                lora_a.weight.grad.copy_(preconditioned_a_grad_f32.to(lora_a.weight.grad.dtype))
                lora_b.weight.grad.copy_(preconditioned_b_grad_f32.to(lora_b.weight.grad.dtype))

                del a_inv_f32, g_inv_f32, grad_a_f32, grad_b_f32, preconditioned_a_grad_f32, preconditioned_b_grad_f32

    def neural_reprojection(self):
        print(f"\nGRITManager: Neural reprojection at step {self.global_step}...")
        initial_params = 0
        final_params = 0

        with torch.no_grad():
            for module in self.optimized_modules:
                if hasattr(module, 'in_features') and hasattr(module, 'out_features'):
                    initial_params += module.r['default'] * (module.in_features + module.out_features)

            for module in self.optimized_modules:
                try:
                    lora_a = module.lora_A['default']
                    lora_b = module.lora_B['default']
                    if lora_a is None or lora_b is None:
                        continue

                    A = lora_a.weight.data.float()
                    B = lora_b.weight.data.float()
                    r = A.shape[0]
                    k = r
                    M = self.a_covs[module].float()

                    if torch.isnan(M).any() or torch.isinf(M).any():
                        final_params += r * (module.in_features + module.out_features)
                        continue

                    min_samples = int(getattr(self.config, 'kfac_min_samples', 128) or 128)
                    if self.num_samples_a.get(module, 0) < min_samples:
                        final_params += r * (module.in_features + module.out_features)
                        continue

                    if self.config.enable_rank_adaptation and r > self.config.min_lora_rank and \
                       self.global_step >= int(getattr(self.config, 'rank_adaptation_start_step', 0) or 0):
                        evals_a, V_a = torch.linalg.eigh(self.a_covs[module].float())
                        order_a = torch.argsort(evals_a, descending=True)
                        V_a = V_a[:, order_a]
                        evals_a = evals_a[order_a]
                        total_energy = torch.sum(evals_a)
                        if total_energy > 1e-6:
                            cumulative_energy = torch.cumsum(evals_a, dim=0) / total_energy
                            k = (cumulative_energy < self.config.rank_adaptation_threshold).sum().item() + 1
                        k = max(k, self.config.min_lora_rank)
                        k = min(k, r)
                        V_a_k = V_a[:, :k]

                        try:
                            g_cov_m = self.g_covs.get(module, None)
                            if getattr(self.config, 'use_two_sided_reprojection', False) and g_cov_m is not None and self.num_samples_g.get(module, 0) >= min_samples:
                                evals_g, V_g = torch.linalg.eigh(g_cov_m.float())
                                order_g = torch.argsort(evals_g, descending=True)
                                V_g_k = V_g[:, order_g][:, :k]
                            else:
                                V_g_k = V_a_k
                        except Exception:
                            V_g_k = V_a_k
                    else:
                        k = min(self.config.reprojection_k, r)
                        evals_a, V_a = torch.linalg.eigh(self.a_covs[module].float())
                        order_a = torch.argsort(evals_a, descending=True)
                        V_a_k = V_a[:, order_a][:, :k]

                        try:
                            g_cov_m = self.g_covs.get(module, None)
                            if getattr(self.config, 'use_two_sided_reprojection', False) and g_cov_m is not None and self.num_samples_g.get(module, 0) >= min_samples:
                                evals_g, V_g = torch.linalg.eigh(g_cov_m.float())
                                order_g = torch.argsort(evals_g, descending=True)
                                V_g_k = V_g[:, order_g][:, :k]
                            else:
                                V_g_k = V_a_k
                        except Exception:
                            V_g_k = V_a_k

                    self._last_Va_k[module] = V_a_k.detach().cpu()
                    self._last_Vg_k[module] = V_g_k.detach().cpu()

                    V_a_k_d = V_a_k.to(device=A.device, dtype=A.dtype, non_blocking=True)
                    V_g_k_d = V_g_k.to(device=B.device, dtype=A.dtype, non_blocking=True)
                    A_proj = V_a_k_d.T @ A
                    B_proj = B @ V_g_k_d
                    A_new = V_a_k_d @ A_proj
                    B_new = B_proj @ V_g_k_d.T
                    lora_a.weight.data.copy_(A_new.to(lora_a.weight.dtype))
                    lora_b.weight.data.copy_(B_new.to(lora_b.weight.dtype))

                    if hasattr(module, 'in_features') and hasattr(module, 'out_features'):
                        final_params += k * (module.in_features + module.out_features)

                except Exception as e:
                    print(f"Error during reprojection for {module.lora_name}: {e}")
                    if hasattr(module, 'in_features') and hasattr(module, 'out_features'):
                        final_params += module.r['default'] * (module.in_features + module.out_features)

        if initial_params > 0:
            param_reduction = initial_params - final_params
            reduction_percent = (param_reduction / initial_params) * 100
            print(f"Neural reprojection: {initial_params:,} -> {final_params:,} ({reduction_percent:.2f}% reduction)")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()


def jit_invert_tensor_pair(a_cov: torch.Tensor, g_cov: torch.Tensor, kfac_damping: float):
    with torch.no_grad():
        base_damping = float(max(kfac_damping, 1e-6))
        a_cov_f = a_cov.float(); a_cov_f = 0.5 * (a_cov_f + a_cov_f.T)
        g_cov_f = g_cov.float(); g_cov_f = 0.5 * (g_cov_f + g_cov_f.T)
        n_a = int(a_cov_f.shape[0]); n_g = int(g_cov_f.shape[0])
        dev = a_cov_f.device
        I_a = torch.eye(n_a, device=dev); I_g = torch.eye(n_g, device=dev)
        scales = [1.0, 3.0, 10.0, 30.0, 100.0, 300.0]
        for s in scales:
            damp_a = float(base_damping * s)
            damp_g = float(base_damping * s)
            L_a, info_a = torch.linalg.cholesky_ex(a_cov_f + damp_a * I_a)
            L_g, info_g = torch.linalg.cholesky_ex(g_cov_f + damp_g * I_g)
            if int(info_a.item()) == 0 and int(info_g.item()) == 0:
                a_inv = torch.cholesky_inverse(L_a).float()
                g_inv = torch.cholesky_inverse(L_g).float()
                return a_inv, g_inv
        return torch.empty(0, device=dev), torch.empty(0, device=dev)


class GritOptimizer(Optimizer):
    """Wrapper that applies GRIT preconditioning before the underlying step."""

    def __init__(self, optimizer: Optimizer, grit_manager: 'GRITManager'):
        self.optimizer = optimizer
        self.grit_manager = grit_manager

    @property
    def state(self):
        return self.optimizer.state

    @property
    def param_groups(self):
        return self.optimizer.param_groups

    @param_groups.setter
    def param_groups(self, value):
        self.optimizer.param_groups = value

    def step(self, closure=None):
        if self.grit_manager.factors_are_ready and not getattr(self.grit_manager, "_preconditioned", False):
            self.grit_manager.precondition_gradients()
        self.optimizer.step(closure)

    def zero_grad(self, set_to_none: bool = False):
        self.optimizer.zero_grad(set_to_none=set_to_none)

    def add_param_group(self, param_group: dict):
        self.optimizer.add_param_group(param_group)

    def state_dict(self):
        return self.optimizer.state_dict()

    def load_state_dict(self, state_dict: dict):
        self.optimizer.load_state_dict(state_dict)

    def __repr__(self):
        return f"GritOptimizer({self.optimizer.__repr__()})"


class GritCallback(TrainerCallback):
    def __init__(self, grit_manager):
        self.grit_manager = grit_manager

    def on_step_end(self, args, state, control, **kwargs):
        last_loss = state.log_history[-1].get("loss") if state.log_history else None
        self.grit_manager.step(loss=last_loss)

    def on_train_end(self, args, state, control, **kwargs):
        try:
            self.grit_manager.log_final_effective_ranks()
        except Exception:
            pass
        try:
            self.grit_manager.log_final_raw_ranks()
        except Exception:
            pass



class LazyRefCOCODataset(torch.utils.data.Dataset):
    """
    Tokenizes on the fly instead of datasets.map().

    Mllama pixel_values are [1 image, 4 tiles, 3, 560, 560] per example
    (~15 MB in fp32). Materializing them into Arrow via .map() is fine for
    32 examples but blows up disk / hits the Arrow serialization overflow at
    8000 (same failure we fixed before with SimpleListDataset). Only the
    `answer` column is scanned up front; images are processed per __getitem__.

    If an example fails to tokenize, the next valid one is returned instead.
    Every returned item carries its OWN refcoco_answers, so prediction <->
    reference alignment in the metrics is preserved.
    """

    def __init__(self, hf_ds, tokenize_fn):
        self.ds = hf_ds
        self.tokenize_fn = tokenize_fn
        answers = hf_ds["answer"]
        self.indices = [
            i for i, a in enumerate(answers)
            if a and str(a[0] if isinstance(a, (list, tuple)) else a).strip()
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        n = len(self.indices)
        for off in range(n):
            j = self.indices[(idx + off) % n]
            out = self.tokenize_fn(self.ds[j])
            if out is not None:
                if off > 0:
                    print(f"[LazyRefCOCODataset] example {self.indices[idx % n]} failed; substituted {j}")
                return out
        raise RuntimeError("No example in this split could be tokenized.")


class GritTrainer(Seq2SeqTrainer):
    """Seq2SeqTrainer subclass that injects GRIT preconditioning and regularizers for RefCOCO (Llama / Mllama)."""

    def __init__(self, processor, grit_manager, eval_answers_buffer, gen_config, eos_token_ids, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.processor = processor
        self.grit_manager = grit_manager
        # Mutable dict shared with make_compute_metrics(): holds the raw 2-4
        # human referring expressions per eval example (popped out of each
        # batch in prediction_step) for multi-reference scoring.
        self.eval_answers_buffer = eval_answers_buffer
        # GRITConfig instance — used to cap generation length at eval.
        self.gen_config = gen_config
        # Llama-3 instruct turns end with <|eot_id|>, not the base EOS.
        self.eos_token_ids = eos_token_ids
        try:
            setattr(self.grit_manager, "_last_grad_clip_cap", float(getattr(self.args, "max_grad_norm", 0.0)))
        except Exception:
            pass
        print("GritTrainer: Initialized with GRIT for Llama-3.2-11B-Vision-Instruct.")

    def create_optimizer_and_scheduler(self, num_training_steps: int):
        super().create_optimizer_and_scheduler(num_training_steps)
        if self.optimizer is not None:
            print("Wrapping the optimizer with GRIT preconditioning logic.")
            self.optimizer = GritOptimizer(self.optimizer, self.grit_manager)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Only real model inputs go to Mllama's forward (see NON_MODEL_KEYS).
        model_inputs = {k: v for k, v in inputs.items() if k not in NON_MODEL_KEYS}
        outputs = model(**model_inputs)
        base_loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
        lambda_k = getattr(self.grit_manager.config, "lambda_kfac", 0.0)
        lambda_r = getattr(self.grit_manager.config, "lambda_reproj", 0.0)

        warmup_steps = int(getattr(self.grit_manager.config, "regularizer_warmup_steps", 0) or 0)
        if warmup_steps > 0:
            prog = min(1.0, max(0.0, self.grit_manager.global_step / float(warmup_steps)))
            lambda_k = float(lambda_k) * prog
            lambda_r = float(lambda_r) * prog

        curv_reg = torch.tensor(0.0, device=base_loss.device)
        reproj_reg = torch.tensor(0.0, device=base_loss.device)

        for module in getattr(self.grit_manager, "optimized_modules", []):
            lora_a = module.lora_A['default'] if 'default' in module.lora_A else None
            lora_b = module.lora_B['default'] if 'default' in module.lora_B else None
            if lora_a is None or lora_b is None:
                continue

            A = lora_a.weight
            B = lora_b.weight

            a_cov = self.grit_manager.a_covs.get(module, None)
            g_cov = self.grit_manager.g_covs.get(module, None)

            if a_cov is not None:
                A_f = A.float()
                a_cov_f = a_cov.to(A_f.device, dtype=A_f.dtype)
                curv_reg = curv_reg + ((a_cov_f @ A_f) * A_f).sum()

            if g_cov is not None:
                B_f = B.float()
                g_cov_f = g_cov.to(B_f.device, dtype=B_f.dtype)
                curv_reg = curv_reg + ((B_f @ g_cov_f) * B_f).sum()

            Va_cache = getattr(self.grit_manager, "_last_Va_k", {})
            Vg_cache = getattr(self.grit_manager, "_last_Vg_k", {})
            V_a_k = Va_cache.get(module, None)
            V_g_k = Vg_cache.get(module, None)

            if V_a_k is None and a_cov is not None:
                try:
                    evals, evecs = torch.linalg.eigh(a_cov.float())
                    order = torch.argsort(evals, descending=True)
                    evecs = evecs[:, order]
                    total = torch.clamp(evals.sum(), min=1e-8)
                    threshold = float(self.grit_manager.config.rank_adaptation_threshold)
                    if self.grit_manager.global_step < int(getattr(self.grit_manager.config, 'rank_adaptation_start_step', 0) or 0):
                        k = min(self.grit_manager.config.reprojection_k, A.shape[0])
                    else:
                        k = int((torch.cumsum(evals, 0) / total < threshold).sum().item()) + 1
                    k = max(min(k, A.shape[0]), self.grit_manager.config.min_lora_rank)
                    V_a_k = evecs[:, :k]

                    if V_g_k is None:
                        try:
                            if (getattr(self.grit_manager.config, 'use_two_sided_reprojection', False)
                                and g_cov is not None
                                and self.grit_manager.num_samples_g.get(module, 0) >= int(getattr(self.grit_manager.config, 'kfac_min_samples', 128) or 128)):
                                evals_g, V_gmat = torch.linalg.eigh(g_cov.float())
                                order_g = torch.argsort(evals_g, descending=True)
                                V_g_k = V_gmat[:, order_g][:, :k]
                            else:
                                V_g_k = V_a_k
                        except Exception:
                            V_g_k = V_a_k
                except Exception:
                    V_a_k = None
                    V_g_k = None

            if V_a_k is not None and V_a_k.numel() > 0:
                A_f = A.float(); B_f = B.float()
                device, dtype = A_f.device, A_f.dtype
                V_a_k_d = V_a_k.to(device=device, dtype=dtype, non_blocking=True)
                V_g_k_d = (V_g_k if V_g_k is not None else V_a_k).to(device=device, dtype=dtype, non_blocking=True)
                P_a = V_a_k_d @ V_a_k_d.T
                I_a = torch.eye(P_a.shape[0], device=device, dtype=dtype)
                P_g = V_g_k_d @ V_g_k_d.T
                I_g = torch.eye(P_g.shape[0], device=device, dtype=dtype)
                A_res = (I_a - P_a) @ A_f
                B_res = B_f @ (I_g - P_g)
                reproj_reg = reproj_reg + (A_res.pow(2).sum() + B_res.pow(2).sum())

        loss = base_loss + lambda_k * curv_reg + lambda_r * reproj_reg
        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        if self.args.n_gpu > 1:
            loss = loss.mean()
        self.accelerator.backward(loss)
        return loss.detach() / self.args.gradient_accumulation_steps

    def optimizer_step(self, *args, **kwargs):
        setattr(self.grit_manager, "_preconditioned", False)
        if getattr(self.grit_manager, "factors_are_ready", False):
            self.grit_manager.precondition_gradients()
            setattr(self.grit_manager, "_preconditioned", True)
        return super().optimizer_step(*args, **kwargs)

    def evaluate(self, *args, **kwargs):
        print("\nClearing VRAM before evaluation...")
        gc.collect()
        torch.cuda.empty_cache()
        self.eval_answers_buffer["answers"] = []
        return super().evaluate(*args, **kwargs)

    def prediction_step(
        self,
        model: nn.Module,
        inputs: Dict[str, torch.Tensor],
        prediction_loss_only: bool,
        ignore_keys: Optional[list] = None,
    ) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor], Optional[torch.Tensor]]:
        """
        Llama / Mllama version.

        inputs["input_ids"] is the FULL chat-templated sequence
            <|begin_of_text|> user header <|image|> instruction <|eot_id|>
            assistant header  answer <|eot_id|>
        produced by ONE processor call, so cross_attention_mask matches it
        position-for-position. The collator RIGHT-pads, so the prompt of
        example i is always input_ids[i, :prompt_length[i]].

        For generation we SLICE (never extend) input_ids, attention_mask AND
        cross_attention_mask to the prompt length, then left-repad. Slicing a
        processor-built mask gives exactly what a prompt-only processor call
        would give (tokenize_refcoco asserts the prompt ids are an exact
        prefix of the full ids), so the text/mask lengths always agree ->
        no device-side assert. pixel_values / aspect_ratio_ids /
        aspect_ratio_mask are per-image and carried over unchanged.

        The teacher-forced eval loss still uses the full [prompt][answer]
        sequence.
        """
        labels = inputs.pop("labels", None)
        has_labels = labels is not None

        # Raw human answers: plain Python lists, never sent to the model.
        ref_answers = inputs.pop("refcoco_answers", None)
        if ref_answers is not None:
            self.eval_answers_buffer["answers"].extend(ref_answers)

        prompt_lengths = inputs.pop("prompt_length", None)

        inputs = self._prepare_inputs(inputs)
        input_ids = inputs["input_ids"]
        input_len = input_ids.shape[1]

        if prompt_lengths is not None:
            attn = inputs["attention_mask"]
            cam = inputs.get("cross_attention_mask", None)
            pad_id = self.processor.tokenizer.pad_token_id
            bsz = input_ids.shape[0]
            p_lens = [int(prompt_lengths[i]) for i in range(bsz)]
            max_p = max(p_lens)

            gen_input_ids = torch.full((bsz, max_p), pad_id, dtype=input_ids.dtype, device=input_ids.device)
            gen_attention_mask = torch.zeros((bsz, max_p), dtype=attn.dtype, device=attn.device)
            gen_cam = None
            if cam is not None:
                gen_cam = torch.zeros((bsz, max_p) + tuple(cam.shape[2:]), dtype=cam.dtype, device=cam.device)

            # Re-pad on the LEFT for batched decoding.
            for i, p in enumerate(p_lens):
                gen_input_ids[i, max_p - p:] = input_ids[i, :p]
                gen_attention_mask[i, max_p - p:] = attn[i, :p]
                if gen_cam is not None:
                    gen_cam[i, max_p - p:] = cam[i, :p]

            gen_inputs = {
                "input_ids": gen_input_ids,
                "attention_mask": gen_attention_mask,
            }
            if gen_cam is not None:
                gen_inputs["cross_attention_mask"] = gen_cam
            for k in ("pixel_values", "aspect_ratio_ids", "aspect_ratio_mask"):
                if k in inputs:
                    gen_inputs[k] = inputs[k]

            gen_input_len = gen_input_ids.shape[1]
        else:
            # Should not happen with tokenize_refcoco; kept as a fallback.
            gen_inputs = {k: v for k, v in inputs.items() if k not in NON_MODEL_KEYS}
            gen_input_len = input_len

        with torch.no_grad():
            if prediction_loss_only:
                loss = None
                if has_labels:
                    inputs_copy = dict(inputs)
                    inputs_copy["labels"] = labels
                    loss = self.compute_loss(model, inputs_copy).detach()
                return (loss, None, None)

            generated_tokens = model.generate(
                **gen_inputs,
                max_new_tokens=self.gen_config.generation_max_new_tokens,
                min_new_tokens=2,
                do_sample=False,
                use_cache=True,
                pad_token_id=self.processor.tokenizer.pad_token_id,
                eos_token_id=self.eos_token_ids,
            )

            # Decoder-only output = [prompt][new tokens]; keep new tokens only.
            generated_tokens = generated_tokens[:, gen_input_len:]

            # Teacher-forced loss on the FULL [prompt][answer] sequence.
            loss = None
            if has_labels:
                inputs_copy = dict(inputs)
                inputs_copy["labels"] = labels
                loss = self.compute_loss(model, inputs_copy).detach()

        if labels is not None:
            if labels.dim() == 3:
                labels = torch.argmax(labels, dim=-1)
            elif labels.dim() == 1:
                batch_size = input_ids.shape[0]
                labels = labels.view(batch_size, -1)

        return loss, generated_tokens, labels


def build_lora_target_regex(config):
    """
    Language tower: q/k/v/o on the Llama text model (self-attn + cross-attn layers).
    Vision tower  : q/k/v only on the Mllama vision encoder.
    A plain list ["q_proj","k_proj","v_proj","o_proj"] would ALSO hit the
    vision encoder's o_proj, so we scope each list to its tower with a regex
    (PEFT does re.fullmatch on module names). Matches both the old
    (language_model.model.layers...) and new (model.language_model.layers...)
    transformers module layouts.
    """
    lang = "|".join(re.escape(m) for m in config.lora_target_modules)
    parts = [rf".*language_model\..*\.({lang})"]
    if config.apply_lora_to_vision:
        vis = "|".join(re.escape(m) for m in config.vision_lora_modules)
        parts.append(rf".*vision_model\..*\.({vis})")
    return "|".join(f"(?:{p})" for p in parts)


def main():
    set_seed(SEED)
    config = GRITConfig()

    print("Loading processor and model for Llama-3.2-11B-Vision-Instruct (RefCOCO)...")
    processor = AutoProcessor.from_pretrained(
        config.model_id,
        token=hf_token,
        trust_remote_code=True,
        padding_side="left",
    )
    tokenizer = processor.tokenizer

    if tokenizer.pad_token is None:
        # Llama-3 ships a dedicated pad token; fall back to EOS only if absent.
        if "<|finetune_right_pad_id|>" in tokenizer.get_vocab():
            tokenizer.pad_token = "<|finetune_right_pad_id|>"
        else:
            tokenizer.pad_token = tokenizer.eos_token
    print(f"Pad token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

    # Stop generation on any Llama-3 end marker (instruct turns end with <|eot_id|>).
    eos_token_ids = []
    for t in [tokenizer.eos_token, "<|eot_id|>", "<|end_of_text|>"]:
        if t is None:
            continue
        tid = tokenizer.convert_tokens_to_ids(t)
        if isinstance(tid, int) and tid >= 0 and tid != tokenizer.unk_token_id and tid not in eos_token_ids:
            eos_token_ids.append(tid)
    print(f"EOS token ids for generation: {eos_token_ids}")

    model = MllamaForConditionalGeneration.from_pretrained(
        config.model_id,
        token=hf_token,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )
    # NOTE: deliberately no device_map="auto" (meta-tensor issues in earlier Llama runs).

    target_regex = build_lora_target_regex(config)
    print(f"LoRA target regex: {target_regex}")

    lora_config = LoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        target_modules=target_regex,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        inference_mode=False,
        use_rslora=True,
    )

    model = get_peft_model(model, lora_config)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    meta_params = [name for name, param in model.named_parameters() if param.device.type == "meta"]
    if meta_params:
        raise RuntimeError(f"Model still has meta parameters: {meta_params[:10]}")
    print("✓ Model has no meta parameters")
    print(f"✓ Model device: {next(model.parameters()).device}")

    # Sanity: LoRA must land on BOTH towers.
    from peft.tuners.lora import LoraLayer
    lora_names = [n for n, m in model.named_modules() if isinstance(m, LoraLayer)]
    n_vis = sum('vision_model' in n for n in lora_names)
    n_lang = sum('language_model' in n for n in lora_names)
    print(f"LoRA modules -> language: {n_lang}, vision: {n_vis}")
    if n_lang == 0 or (config.apply_lora_to_vision and n_vis == 0):
        raise RuntimeError("LoRA target regex did not match both towers — check module names.")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")
    print(f"Percentage: {100 * trainable_params / total_params:.3f}%")

    model.enable_input_require_grads()
    model.config.use_cache = False
    if getattr(model.config, "text_config", None) is not None:
        model.config.text_config.use_cache = False

    print("Loading RefCOCO dataset with splits...")
    full_dataset = load_dataset(config.dataset_name)

    def _take(split, n):
        ds = full_dataset[split].shuffle(seed=SEED)
        return ds.select(range(min(n, len(ds))))

    train_dataset = _take("val", config.train_size)
    val_dataset   = _take("testA", config.val_size)
    test_dataset  = _take("testB", config.test_size)

    print(f"Final splits → Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    _debug_state = {"printed": False}

    def tokenize_refcoco(example):
        """
        Llama / Mllama multimodal tokenization.

          1) prompt_text = chat template of [user: <image> + instruction],
             add_generation_prompt=True  (ends with the assistant header).
          2) full_text   = chat template of [user ..., assistant: answer]
             (ends with answer + <|eot_id|>).
          3) ONE processor call on (image, full_text) -> input_ids,
             attention_mask, pixel_values, aspect_ratio_ids,
             aspect_ratio_mask, cross_attention_mask, all mutually consistent.
          4) prompt_length P = len(tokenized prompt_text). We ASSERT the
             prompt ids are an exact prefix of the full ids, then
             labels = -100 on [:P], answer ids (+ <|eot_id|>) on [P:].

        The chat template already emits <|begin_of_text|>, so the processor
        is called with add_special_tokens=False (no double BOS).
        We never manually concatenate answer ids onto processor tensors.
        """
        try:
            if 'image' not in example or example['image'] is None:
                return None

            image = load_image_from_source(example['image'])
            if image is None or not isinstance(image, Image.Image):
                return None
            if image.mode != 'RGB':
                image = image.convert('RGB')

            # Draw the referred region (red box at the GT bbox), as in the Gemma run.
            image = draw_region_indicator(image, example.get('bbox'))

            # Train on the first human expression; all 2-4 kept for eval metrics.
            answers_list = example.get('answer') or []
            answer = (answers_list[0] if answers_list else '').strip()
            if not answer:
                return None

            user_turn = {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": REFCOCO_INSTRUCTION},
                ],
            }
            assistant_turn = {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            }

            prompt_text = processor.apply_chat_template(
                [user_turn], add_generation_prompt=True, tokenize=False
            )
            full_text = processor.apply_chat_template(
                [user_turn, assistant_turn], add_generation_prompt=False, tokenize=False
            )

            if "<|image|>" not in prompt_text:
                print("ERROR: <|image|> token missing from chat template output.")
                return None

            full = processor(
                images=[[image]],
                text=[full_text],
                add_special_tokens=False,
                return_tensors="pt",
            )

            input_ids = full["input_ids"][0].long()
            attention_mask = full["attention_mask"][0].long()

            prompt_ids = tokenizer(
                prompt_text, add_special_tokens=False, return_tensors="pt"
            )["input_ids"][0].long()
            P = int(prompt_ids.shape[0])

            if P >= input_ids.shape[0] or not torch.equal(input_ids[:P], prompt_ids):
                print("WARNING: prompt ids are not a prefix of full ids — skipping example.")
                return None

            if input_ids.shape[0] > config.max_length:
                return None

            labels = input_ids.clone()
            labels[:P] = config.ignore_index

            cross_attention_mask = full["cross_attention_mask"][0]  # [seq, n_images, n_tiles]
            if cross_attention_mask.shape[0] != input_ids.shape[0]:
                print("WARNING: cross_attention_mask / input_ids length mismatch — skipping example.")
                return None

            output = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
                "cross_attention_mask": cross_attention_mask,
                "pixel_values": full["pixel_values"][0],            # [n_images, n_tiles, 3, 560, 560]
                "aspect_ratio_ids": full["aspect_ratio_ids"][0],    # [n_images]
                "aspect_ratio_mask": full["aspect_ratio_mask"][0],  # [n_images, n_tiles]
                # metadata (stripped before the model sees the batch)
                "prompt_length": P,
                "refcoco_answers": extract_raw_answers(example),
            }

            if not _debug_state["printed"]:
                _debug_state["printed"] = True
                tgt = input_ids[P:]
                print("\n[Tokenization Debug] ---------------------------------")
                print(f"Full seq len: {input_ids.shape[0]} | prompt len: {P} | target len: {tgt.shape[0]}")
                print(f"Prompt tail : {tokenizer.decode(input_ids[max(0, P - 8):P])!r}")
                print(f"Target (raw): {tokenizer.decode(tgt)!r}")
                print(f"pixel_values {tuple(output['pixel_values'].shape)} | "
                      f"aspect_ratio_ids {tuple(output['aspect_ratio_ids'].shape)} | "
                      f"aspect_ratio_mask {tuple(output['aspect_ratio_mask'].shape)} | "
                      f"cross_attention_mask {tuple(cross_attention_mask.shape)}")
                print("-----------------------------------------------------\n")

            return output

        except Exception as e:
            print(f"Error processing example: {e}")
            import traceback
            traceback.print_exc()
            return None

    print("Building lazy datasets (on-the-fly tokenization)...")
    tokenized_train_dataset = LazyRefCOCODataset(train_dataset, tokenize_refcoco)
    tokenized_val_dataset = LazyRefCOCODataset(val_dataset, tokenize_refcoco)
    tokenized_test_dataset = LazyRefCOCODataset(test_dataset, tokenize_refcoco)

    train_len = len(tokenized_train_dataset)
    val_len = len(tokenized_val_dataset)
    test_len = len(tokenized_test_dataset)

    if train_len == 0:
        raise ValueError(
            "No training samples after processing. "
            "Possible causes: images not loading correctly, dataset format incompatible, "
            "or all examples failed validation."
        )

    # Fail fast: tokenize one example now (also prints the debug block above).
    _probe = tokenized_train_dataset[0]
    assert _probe is not None

    print(f"\nSuccessfully prepared datasets:")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")

    eff_batch = config.batch_size * config.gradient_accumulation_steps
    total_steps = math.ceil(train_len / eff_batch) * config.num_epochs

    print(f"\nTraining Configuration:")
    print(f"Total training steps: {total_steps}")
    print(f"Effective batch size: {eff_batch}")
    print(f"Steps per epoch: {math.ceil(train_len / eff_batch)}")

    training_args = Seq2SeqTrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, f"GRIT_llama4_seed{SEED}_VG"),

        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,

        num_train_epochs=config.num_epochs,
        learning_rate=config.learning_rate,

        bf16=(config.precision == "bf16"),
        fp16=(config.precision == "fp16"),

        max_grad_norm=1.5,

        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},

        dataloader_num_workers=config.num_workers,
        dataloader_pin_memory=config.pin_memory,
        dataloader_drop_last=config.drop_last,

        remove_unused_columns=False,
        report_to="none",

        save_strategy="no",
        eval_strategy="no",

        logging_steps=config.logging_steps,
        warmup_steps=config.warmup_steps,

        optim="adamw_torch",

        # Reproducibility
        seed=SEED,
        data_seed=SEED,

        #prediction_loss_only=False,
        #predict_with_generate=True,
    )

    print(f"\nInitializing GRIT on device: {device}")
    grit_manager = GRITManager(model, config, device)

    pixel_dtype = torch.bfloat16 if config.precision == "bf16" else (
        torch.float16 if config.precision == "fp16" else torch.float32
    )

    def _pad_tiles(t, target_tiles, tile_dim):
        cur = t.shape[tile_dim]
        if cur == target_tiles:
            return t
        pad_shape = list(t.shape)
        pad_shape[tile_dim] = target_tiles - cur
        return torch.cat([t, t.new_zeros(pad_shape)], dim=tile_dim)

    def collate_fn(batch):
        """
        Mllama collator.
        - RIGHT-pads input_ids / attention_mask / labels (so prediction_step
          can take the prompt as input_ids[i, :prompt_length]).
        - Pads cross_attention_mask along the sequence dim with zeros so it
          always has exactly the same length as input_ids.
        - Stacks the per-image vision tensors.
        """
        batch = [b for b in batch if b is not None and isinstance(b, dict) and 'input_ids' in b]
        if len(batch) == 0:
            raise RuntimeError("Empty batch in collate_fn.")

        pad_id = tokenizer.pad_token_id

        input_ids = torch.nn.utils.rnn.pad_sequence(
            [b['input_ids'] for b in batch], batch_first=True, padding_value=pad_id
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [b['attention_mask'] for b in batch], batch_first=True, padding_value=0
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            [b['labels'] for b in batch], batch_first=True, padding_value=config.ignore_index
        )
        max_len = input_ids.shape[1]

        # The processor pads to max_image_tiles already; this is just a guard.
        max_tiles = max(b['pixel_values'].shape[1] for b in batch)

        cams, pvs, ar_ids, ar_masks = [], [], [], []
        for b in batch:
            cam = _pad_tiles(b['cross_attention_mask'], max_tiles, tile_dim=-1)
            seq_pad = max_len - cam.shape[0]
            if seq_pad > 0:
                cam = torch.cat([cam, cam.new_zeros((seq_pad,) + tuple(cam.shape[1:]))], dim=0)
            cams.append(cam)
            pvs.append(_pad_tiles(b['pixel_values'], max_tiles, tile_dim=1))
            ar_ids.append(b['aspect_ratio_ids'])
            ar_masks.append(_pad_tiles(b['aspect_ratio_mask'], max_tiles, tile_dim=-1))

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'cross_attention_mask': torch.stack(cams, dim=0),          # [B, seq, n_img, n_tiles]
            'pixel_values': torch.stack(pvs, dim=0).to(pixel_dtype),  # [B, n_img, n_tiles, 3, H, W]
            'aspect_ratio_ids': torch.stack(ar_ids, dim=0),            # [B, n_img]
            'aspect_ratio_mask': torch.stack(ar_masks, dim=0),         # [B, n_img, n_tiles]
            # metadata — popped / stripped before the model sees the batch
            'refcoco_answers': [b.get('refcoco_answers', []) for b in batch],
            'prompt_length': torch.tensor([int(b['prompt_length']) for b in batch], dtype=torch.long),
        }

    def make_compute_metrics(processor, eval_answers_buffer):
        import re
        import numpy as np
        from rouge_score import rouge_scorer
        from nltk.translate.meteor_score import meteor_score
        from bert_score import score as bert_score
        from collections import defaultdict
        import nltk
        nltk.download('wordnet', quiet=True)
        nltk.download('omw-1.4', quiet=True)

        # PATCH FOR MOVERSCORE COMPATIBILITY
        import transformers.tokenization_utils_base as tub
        if not hasattr(tub.PreTrainedTokenizerBase, '_moverscore_patched'):
            original_init = tub.PreTrainedTokenizerBase.__init__
            def patched_init(self, *args, **kwargs):
                original_init(self, *args, **kwargs)
                if not hasattr(self, 'max_len'):
                    self.max_len = self.model_max_length
            tub.PreTrainedTokenizerBase.__init__ = patched_init
            tub.PreTrainedTokenizerBase._moverscore_patched = True
            print("✓ Applied moverscore compatibility patch")

        # Now import moverscore
        moverscore_available = False
        try:
            from moverscore_v2 import get_idf_dict, word_mover_score
            moverscore_available = True
            print("✓ MoverScore loaded successfully")
        except ImportError as e:
            print(f"✗ MoverScore not available: {e}")

        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

        tok = processor.tokenizer

        def extract_answer(text):
            """Clean Llama-3 output safely without truncating content."""

            if text is None:
                return ""

            # Remove Llama-3 special tokens (normally already skipped by decode)
            text = text.replace("<|begin_of_text|>", "")
            text = text.replace("<|end_of_text|>", "")
            text = text.replace("<|start_header_id|>", "")
            text = text.replace("<|end_header_id|>", "")
            text = text.replace("<|eot_id|>", "")
            text = text.replace("<|image|>", "")

            # Remove an accidental leading assistant header if it survives decoding
            text = re.sub(r"^\s*assistant\s*", "", text, flags=re.IGNORECASE)

            # Normalize whitespace
            text = re.sub(r"\s+", " ", text).strip()

            return text

        # Official VQA eval answer processing (from the VQA project's
        # vqaEval.py), used so exact-match comparisons for accuracy (and
        # the text-similarity metrics below) aren't broken by trailing
        # punctuation, articles, or number-word vs. digit mismatches.
        _vqa_contractions = {
            "aint": "ain't", "arent": "aren't", "cant": "can't", "couldve": "could've",
            "couldnt": "couldn't", "didnt": "didn't", "doesnt": "doesn't", "dont": "don't",
            "hadnt": "hadn't", "hasnt": "hasn't", "havent": "haven't", "hed": "he'd",
            "hes": "he's", "im": "i'm", "ive": "i've", "isnt": "isn't", "its": "it's",
            "lets": "let's", "shant": "shan't", "shes": "she's", "shouldve": "should've",
            "shouldnt": "shouldn't", "thats": "that's", "theres": "there's", "theyd": "they'd",
            "theyre": "they're", "theyve": "they've", "wasnt": "wasn't", "wed": "we'd",
            "were": "we're", "weve": "we've", "werent": "weren't", "whats": "what's",
            "wheres": "where's", "whos": "who's", "wont": "won't", "wouldve": "would've",
            "wouldnt": "wouldn't", "youd": "you'd", "youre": "you're", "youve": "you've",
        }
        _vqa_number_words = {
            "none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
            "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
        }
        _vqa_articles = {"a", "an", "the"}
        _vqa_punct = [';', '/', '[', ']', '"', '{', '}', '(', ')', '=', '+', '\\',
                      '_', '-', '>', '<', '@', '`', ',', '?', '!', '.', ':']

        def normalize(s):
            """Official VQA-style answer normalization."""
            s = s.lower().strip()
            # Remove punctuation entirely (rather than spacing it out),
            # so "brown." normalizes to "brown", not "brown .".
            for p in _vqa_punct:
                s = s.replace(p, ' ')
            s = re.sub(r"\s+", " ", s).strip()

            words = []
            for word in s.split():
                word = _vqa_number_words.get(word, word)
                if word in _vqa_articles:
                    continue
                word = _vqa_contractions.get(word, word)
                words.append(word)
            return ' '.join(words).strip()

        def safe_ids(arr):
            """
            Llama-safe id cleanup before decoding (predictions AND labels).
            - Any negative id (-100 label mask, or the -100 the Trainer uses to
              pad generations across eval batches) -> pad id. Clipping -100 to
              0 instead would decode as "!" in Llama's vocab.
            - Upper bound is len(tokenizer) (128k base + added special tokens),
              NOT tokenizer.vocab_size (base only): clipping to vocab_size-1
              would turn <|eot_id|> (128009) into a real text token (127999)
              that skip_special_tokens can no longer remove.
            """
            arr = np.asarray(arr, dtype=np.int64).copy()
            pad_token_id = tok.pad_token_id
            arr[arr < 0] = pad_token_id
            arr[arr >= len(tok)] = pad_token_id
            return arr

        def compute_refcoco_metrics(p: EvalPrediction):
            pred_ids = safe_ids(p.predictions)
            labels = safe_ids(p.label_ids)

            try:
                pred_texts_raw = tok.batch_decode(pred_ids, skip_special_tokens=True)
                label_texts = tok.batch_decode(labels, skip_special_tokens=True)
            except Exception as e:
                print(f"[Metrics] Decode error: {e}")
                return {"rouge_l": 0.0, "meteor": 0.0, "bertscore_f1": 0.0, "moverscore": 0.0, "accuracy": 0.0}

            pred_texts = [extract_answer(t) for t in pred_texts_raw]
            pred_norm = [normalize(t) for t in pred_texts]
            label_texts = [extract_answer(t) for t in label_texts]
            label_norm = [normalize(t) for t in label_texts]

            for i in range(min(3, len(pred_norm))):
                print(f"\n[Sample {i+1}]")
                print(f"Generated: {pred_texts[i][:100]}")
                print(f"Reference: {label_texts[i][:100]}")

            # FIX (root of the "half-baked scores" issue): everything below
            # used to be scored against `label_norm`, which is decoded from
            # the `labels` tensor — and `labels` only ever encodes
            # answers_list[0] (see tokenize_refcoco), i.e. ONE arbitrary human
            # reference out of the 2-4 RefCOCO collects per region. A
            # correct prediction that happens to match reference #2 or #3
            # instead of #1 scored as a near-total miss on ROUGE-L/METEOR.
            #
            # lmms-eval's own aggregation for this exact dataset
            # (refcoco_aggregation_result) instead feeds ALL of
            # doc["answer"] in as separate reference captions for the same
            # image_id and lets pycocoevalcap's scorers take the best
            # match across the full reference set. We reproduce that here
            # using eval_answers_buffer (already populated by
            # GritTrainer.prediction_step with the raw 2-4 answers per
            # example) instead of the single-answer label_norm.
            raw_answers_list = eval_answers_buffer.get("answers", [])
            use_multi_ref = len(raw_answers_list) == len(pred_norm)

            if use_multi_ref:
                norm_refs_per_example = [
                    [normalize(a) for a in raw_answers if a] or [l]
                    for raw_answers, l in zip(raw_answers_list, label_norm)
                ]
            else:
                print(
                    f"[Metrics] WARNING: buffered answers ({len(raw_answers_list)}) "
                    f"!= predictions ({len(pred_norm)}); falling back to "
                    f"single-reference scoring against the training answer."
                )
                norm_refs_per_example = [[l] for l in label_norm]

            valid_pairs = [
                (p, refs) for p, refs in zip(pred_norm, norm_refs_per_example)
                if p and refs and len(p.split()) > 0
            ]

            # ROUGE-L — max over this example's reference set.
            rouge_ls = []
            for p, refs in valid_pairs:
                best = max(
                    (scorer.score(r, p)['rougeL'].fmeasure for r in refs if r),
                    default=None,
                )
                if best is not None:
                    rouge_ls.append(best)
            rouge_l = np.mean(rouge_ls) if rouge_ls else 0.0

            # METEOR — nltk's meteor_score natively supports multiple
            # references and takes the max internally, so just pass the
            # full reference set instead of a single one.
            meteors = []
            for p, refs in valid_pairs:
                ref_token_lists = [r.split() for r in refs if r]
                if ref_token_lists:
                    meteors.append(meteor_score(ref_token_lists, p.split()))
            meteor = np.mean(meteors) if meteors else 0.0

            # BERTscore / MoverScore need one (pred, ref) string pair per
            # call, so we flatten pred-vs-each-of-its-refs into parallel
            # lists, score everything in one batched call, then take the
            # max back per original example.
            flat_preds, flat_refs, group_idx = [], [], []
            for i, (p, refs) in enumerate(valid_pairs):
                for r in refs:
                    if r:
                        flat_preds.append(p)
                        flat_refs.append(r)
                        group_idx.append(i)

            def _max_per_group(flat_scores, n_groups):
                best = [None] * n_groups
                for idx, s in zip(group_idx, flat_scores):
                    if best[idx] is None or s > best[idx]:
                        best[idx] = s
                return [b for b in best if b is not None]

            # BERTscore
            if flat_preds and flat_refs:
                try:
                    P, R, F1 = bert_score(
                        flat_preds,
                        flat_refs,
                        lang="en",
                        model_type="bert-base-uncased",
                        rescale_with_baseline=False
                    )
                    per_ex_best = _max_per_group(F1.tolist(), len(valid_pairs))
                    bertscore_f1 = float(np.mean(per_ex_best)) if per_ex_best else 0.0
                except Exception as e:
                    print(f"[Metrics] BERTScore error: {e}")
                    bertscore_f1 = 0.0
            else:
                bertscore_f1 = 0.0

            # Accuracy — FIX: the original VQA leave-one-out soft-accuracy
            # formula is designed for VQA's setting (short, closed-set
            # answers like "umpire", ~10 annotators) and doesn't transfer
            # well to RefCOCO's setting (free-form referring expressions,
            # only 2-4 annotators):
            #   1) The "/3" threshold caps max achievable accuracy below 1.0
            #      even for a perfect prediction with only 2-4 references
            #      (e.g. ceiling ≈ 0.67 with 3 refs).
            #   2) It requires an EXACT string match. A correct caption like
            #      "this is an umpire" will never exactly equal the
            #      reference "umpire", so accuracy stays near 0 regardless
            #      of how good the model actually is. This is why standard
            #      RefCOCO generation papers report METEOR/CIDEr/ROUGE-L/
            #      BLEU rather than "accuracy" at all — free-form text
            #      doesn't have a single correct string to match against.
            #
            # Fix: use a relaxed containment match instead — a prediction
            # counts as correct for an example if it exactly matches, or
            # contains, or is contained by, any one of that example's human
            # reference expressions (after normalization). This still
            # requires real word-level overlap with a genuine reference (so
            # it isn't free credit), but no longer structurally punishes a
            # correct model for writing "this is an umpire" instead of
            # exactly "umpire", and isn't capped by a VQA-specific ceiling.
            # Treat this as a coarse correct/incorrect signal — ROUGE-L /
            # METEOR / BERTScore / MoverScore remain the primary metrics
            # for judging caption quality.
            # (raw_answers_list was already fetched above for the
            # multi-reference NLG metrics — reused here as-is.)

            print(f"\n[Accuracy Debug] buffered answer-sets: {len(raw_answers_list)} | predictions: {len(pred_norm)}")
            for i in range(min(3, len(raw_answers_list), len(pred_norm))):
                print(f"[Accuracy Debug] Sample {i+1}")
                print(f"  Prediction (normalized): {pred_norm[i]!r}")
                print(f"  Raw human answers       : {raw_answers_list[i]}")
                print(f"  Normalized human answers: {[normalize(a) for a in raw_answers_list[i] if a]}")

            def relaxed_match(pred, ref):
                if not pred or not ref:
                    return False
                if pred == ref:
                    return True
                # Require at least a few characters of real overlap so a
                # single short common word (e.g. "the") can't trivially
                # match — both strings are already normalized (articles
                # stripped, lowercased) at this point.
                if len(ref) >= 3 and ref in pred:
                    return True
                if len(pred) >= 3 and pred in ref:
                    return True
                return False

            if not use_multi_ref:
                # Warning already printed above when building
                # norm_refs_per_example — just apply the same fallback here.
                matches = [1.0 if relaxed_match(p, l) else 0.0 for p, l in zip(pred_norm, label_norm) if l]
                accuracy = float(np.mean(matches)) if matches else 0.0
            else:
                per_question_accs = []
                for pred, raw_answers in zip(pred_norm, raw_answers_list):
                    norm_answers = [normalize(a) for a in raw_answers if a]
                    if not norm_answers:
                        continue
                    is_correct = any(relaxed_match(pred, ref) for ref in norm_answers)
                    per_question_accs.append(1.0 if is_correct else 0.0)
                accuracy = float(np.mean(per_question_accs)) if per_question_accs else 0.0

            # MoverScore — same flatten-then-max-per-example approach as
            # BERTScore above, using flat_preds/flat_refs/group_idx built
            # earlier from the FULL reference set (not just answers_list[0]).
            moverscore_val = 0.0
            if moverscore_available and flat_preds and flat_refs:
                try:
                    from moverscore_v2 import get_idf_dict, word_mover_score

                    idf_dict_ref = get_idf_dict(flat_refs)
                    idf_dict_hyp = get_idf_dict(flat_preds)

                    scores = word_mover_score(
                        flat_refs,
                        flat_preds,
                        idf_dict_ref,
                        idf_dict_hyp,
                        stop_words=[],
                        n_gram=1,
                        remove_subwords=True
                    )

                    if scores and len(scores) > 0:
                        per_ex_best = _max_per_group(scores, len(valid_pairs))
                        moverscore_val = float(np.mean(per_ex_best)) if per_ex_best else 0.0
                        print(f"[MoverScore] Score: {moverscore_val:.4f}")

                except Exception as e:
                    print(f"[MoverScore] Error: {e}")
                    import traceback
                    traceback.print_exc()

            return {
                "rouge_l": float(rouge_l),
                "meteor": float(meteor),
                "bertscore_f1": float(bertscore_f1),
                "moverscore": float(moverscore_val),
                "accuracy": accuracy,
            }

        return compute_refcoco_metrics

    # Shared mutable buffer: GritTrainer.prediction_step fills it with the
    # raw per-question human answers for each eval batch; compute_metrics
    # reads it to compute the multi-reference RefCOCO metrics above.
    eval_answers_buffer = {"answers": []}

    trainer = GritTrainer(
        processor=processor,
        grit_manager=grit_manager,
        eval_answers_buffer=eval_answers_buffer,
        gen_config=config,
        eos_token_ids=eos_token_ids,
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=collate_fn,
        compute_metrics=make_compute_metrics(processor, eval_answers_buffer),
        callbacks=[GritCallback(grit_manager)],
    )

    print("\n" + "="*80)
    print("STARTING GRIT TRAINING FOR LLAMA-3.2-11B-VISION-INSTRUCT ON REFCOCO")
    print("="*80)
    print(f"Model: {config.model_id}")
    print(f"Dataset: {config.dataset_name}")
    print(f"LoRA rank: {config.lora_rank}")
    print(f"Batch size: {config.batch_size} x {config.gradient_accumulation_steps} = {eff_batch}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Precision: {config.precision}")
    print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL'}")
    print(f"K-FAC update freq: {config.kfac_update_freq} steps (min samples {config.kfac_min_samples})")
    print(f"NG preconditioning warmup: {config.ng_warmup_steps} steps")
    print(f"Reprojection freq / warmup: {config.reprojection_freq} / {config.reprojection_warmup_steps} steps")
    print(f"Vision LoRA: {'Enabled' if config.apply_lora_to_vision else 'Disabled'}")
    print(f"Rank adaptation: {'Enabled' if config.enable_rank_adaptation else 'Disabled'}")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")
    print(f"Output dir: {OUTPUT_DIR}")
    print("="*80 + "\n")

    try:
        print("Beginning training...")
        trainer.train()

        print("\n" + "="*80)
        print("FINAL TEST EVALUATION")
        print("="*80)

        test_results = trainer.evaluate(eval_dataset=tokenized_test_dataset)

        result = {
            "method": "GRIT",
            "model": config.model_id,
            "task": "RefCOCO",
            "seed": SEED,

            "train_size": train_len,
            "val_size": val_len,
            "test_size": test_len,

            "epochs": config.num_epochs,
            "batch_size": config.batch_size,
            "gradient_accumulation_steps": config.gradient_accumulation_steps,
            "effective_batch_size": eff_batch,
            "optimizer_steps": total_steps,

            "learning_rate": config.learning_rate,
            "precision": config.precision,

            "lora_rank": config.lora_rank,
            "lora_alpha": config.lora_alpha,
            "lora_dropout": config.lora_dropout,

            "language_target_modules": config.lora_target_modules,
            "vision_target_modules": config.vision_lora_modules,
            "lora_target_regex": target_regex,
            "smoke_test": SMOKE_TEST,

            "kfac_update_freq": config.kfac_update_freq,
            "kfac_min_samples": config.kfac_min_samples,
            "kfac_damping": config.kfac_damping,
            "grit_cov_update_freq": config.grit_cov_update_freq,
            "ng_warmup_steps": config.ng_warmup_steps,
            "regularizer_warmup_steps": config.regularizer_warmup_steps,
            "reprojection_freq": config.reprojection_freq,
            "reprojection_warmup_steps": config.reprojection_warmup_steps,
            "rank_adaptation_start_step": config.rank_adaptation_start_step,

            "test_loss": float(test_results["eval_loss"]),
            "test_rouge_l": float(test_results["eval_rouge_l"]),
            "test_meteor": float(test_results["eval_meteor"]),
            "test_bertscore_f1": float(test_results["eval_bertscore_f1"]),
            "test_moverscore": float(test_results["eval_moverscore"]),
            "test_accuracy": float(test_results["eval_accuracy"]),
        }
        result_path = os.path.join(OUTPUT_DIR, f"{RUN_NAME}.json")

        with open(result_path, "w") as f:
            json.dump(result, f, indent=2)

        print(f"Results saved to: {result_path}")

        print(f"Test Loss          : {test_results['eval_loss']:.5f}")
        print(f"Test ROUGE-L       : {test_results['eval_rouge_l']:.4f}")
        print(f"Test METEOR        : {test_results['eval_meteor']:.4f}")
        print(f"Test BERTscore F1  : {test_results['eval_bertscore_f1']:.4f}")
        print(f"Test MoverScore    : {test_results['eval_moverscore']:.4f}")
        print(f"Test Accuracy      : {test_results['eval_accuracy']:.4f}")

        print("\n" + "="*80)
        print("Saving final model...")
        output_dir = os.path.join(OUTPUT_DIR, "final_refcoco_grit_llama11b_model")
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")

        print("\n" + "="*80)
        print("TRAINING COMPLETED SUCCESSFULLY")
        print("="*80 + "\n")

    except KeyboardInterrupt:
        print("\n" + "="*80)
        print("TRAINING INTERRUPTED BY USER")
        print("="*80)
        print("Saving checkpoint...")
        try:
            ckpt_dir = os.path.join(OUTPUT_DIR, "interrupted_checkpoint_refcoco_grit_llama11b")
            model.save_pretrained(ckpt_dir)
            processor.save_pretrained(ckpt_dir)
            print(f"Checkpoint saved to {ckpt_dir}")
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    except Exception as e:
        print("\n" + "="*80)
        print("TRAINING FAILED")
        print("="*80)
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        raise

    finally:
        print("\nCleaning up...")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Cleanup complete")


if __name__ == "__main__":
    main()


All experiment outputs will be saved in:
/GRIT_llama11b_seed41_RefCOCO_smoke
Loading processor and model for Llama-3.2-11B-Vision-Instruct (RefCOCO)...
Pad token: '<|finetune_right_pad_id|>' (id=128004)
EOS token ids for generation: [128009, 128001]


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

LoRA target regex: (?:.*language_model\..*\.(q_proj|k_proj|v_proj|o_proj))|(?:.*vision_model\..*\.(q_proj|k_proj|v_proj))
✓ Model has no meta parameters
✓ Model device: cuda:0
LoRA modules -> language: 160, vision: 120
Trainable params: 10,977,280
Total params: 10,681,198,115
Percentage: 0.103%
Loading RefCOCO dataset with splits...
Final splits → Train: 32, Val: 4, Test: 4
Building lazy datasets (on-the-fly tokenization)...

[Tokenization Debug] ---------------------------------
Full seq len: 45 | prompt len: 38 | target len: 7
Prompt tail : ' extra text.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
Target (raw): 'top bed from the left side<|eot_id|>'
pixel_values (1, 4, 3, 560, 560) | aspect_ratio_ids (1,) | aspect_ratio_mask (1, 4) | cross_attention_mask (45, 1, 4)
-----------------------------------------------------


Successfully prepared datasets:
Training samples: 32
Validation samples: 4
Test samples: 4

Training Configuration:
Total training steps: 24
Effective

/venv/main/lib/python3.12/site-packages/torch/nn/modules/module.py:1790: FutureWarning: `hidden_state` is deprecated and will be removed in version v5.20 for `MllamaVisionEncoderLayer.forward`. Use `hidden_states` instead.
  return forward_call(*args, **kwargs)


... (18 outputs left out)

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Accuracy Debug] buffered answer-sets: 4 | predictions: 4
[Accuracy Debug] Sample 1
  Prediction (normalized): 'red box'
  Raw human answers       : ['Pick 1 of left 6', 'red cake', 'CLICK ANY ONE BESIDE 1ST AND 2ND RIGHT SIDE']
  Normalized human answers: ['pick 1 of left 6', 'red cake', 'click any 1 beside 1st and 2nd right side']
[Accuracy Debug] Sample 2
  Prediction (normalized): 'red van'
  Raw human answers       : ['Red truck', 'red van', 'Red van being towed']
  Normalized human answers: ['red truck', 'red van', 'red van being towed']
[Accuracy Debug] Sample 3
  Prediction (normalized): 'orange fruit'
  Raw human answers       : ['center lemon that is on top of the aaple', 'oranges closest to banana middle', 'orange directly ontop of apple']
  Normalized human answers: ['center lemon that is on top of aaple', 'oranges closest to banana middle', 'orange directly ontop of apple']
[MoverScore] Score: 0.6624


Training Loss,Validation Loss,Epoch,Rouge L,Meteor,Bertscore F1,Moverscore,Accuracy
2.489556,3.393258,3,0.613095,0.389260,0.792753,0.662381,0.500000


Results saved to: /GRIT_llama11b_seed41_RefCOCO_smoke/GRIT_llama11b_seed41_RefCOCO_smoke.json
Test Loss          : 3.39326
Test ROUGE-L       : 0.6131
Test METEOR        : 0.3893
Test BERTscore F1  : 0.7928
Test MoverScore    : 0.6624
Test Accuracy      : 0.5000

Saving final model...
Model saved to /GRIT_llama11b_seed41_RefCOCO_smoke/final_refcoco_grit_llama11b_model

TRAINING COMPLETED SUCCESSFULLY


Cleaning up...
Cleanup complete
